# Road Accident Severity Prediction – Business Analytics

This notebook contains the preprocessing, exploratory visualization, analytics/modeling, evaluation and outputs used for the case study report.

**Important:** The saved pilot model uses the 290-record cleaned pilot dataset. The larger scraped corpus is article-level and is not assumed to contain one unique row per accident.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

plt.rcParams["figure.figsize"] = (8, 5)


## 1. Load the cleaned pilot dataset

In [ ]:
pilot = pd.read_csv("data/cleaned_pilot_dataset.csv", low_memory=False)

print("Pilot dataset shape:", pilot.shape)
display(pilot.head())


## 2. Data quality and target distribution

The pilot dataset contains only records with non-unknown severity and clear single-state evidence. Death and injury counts are not used as model predictors because they would create target leakage.


In [ ]:
print("Missing values in selected modeling columns:")
display(pilot[[
    "state", "time_of_day", "severity",
    "rain_or_wet", "fog", "night_evidence", "highway",
    "bus_involved", "lorry_truck", "car_involved", "two_wheeler",
    "head_on", "rear_end", "collision"
]].isna().sum())

print("\nSeverity distribution:")
display(pilot["severity"].value_counts())


## 3. Exploratory Analysis – Severity

In [ ]:
pilot["severity"].value_counts().reindex(
    ["Fatal", "Serious", "Minor/Reported Injury"]
).plot(kind="bar")
plt.title("Pilot Modeling Dataset: Severity Classes")
plt.xlabel("Severity class")
plt.ylabel("Records")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 4. Exploratory Analysis – State

In [ ]:
pilot["state"].value_counts().plot(kind="bar")
plt.title("Pilot Dataset: State Distribution")
plt.xlabel("State")
plt.ylabel("Records")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 5. Exploratory Analysis – Time of Day

In [ ]:
pilot["time_of_day"].value_counts().reindex(
    ["Morning", "Afternoon", "Evening", "Night", "Unknown"]
).plot(kind="bar")
plt.title("Pilot Dataset: Reported Time-of-Day Evidence")
plt.xlabel("Time category")
plt.ylabel("Records")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 6. Accident-related factor evidence

In [ ]:
factor_cols = [
    "rain_or_wet", "fog", "night_evidence", "highway",
    "bus_involved", "lorry_truck", "car_involved",
    "two_wheeler", "head_on", "rear_end", "collision"
]
factor_counts = pilot[factor_cols].sum().sort_values()
factor_counts.plot(kind="barh")
plt.title("Extracted Accident-Related Factor Evidence")
plt.xlabel("Pilot records")
plt.ylabel("Factor")
plt.tight_layout()
plt.show()


## 7. Feature preparation

Categorical variables are one-hot encoded. Binary contextual indicators are passed through directly. Severity is the target variable.

Deaths and injuries are deliberately excluded from the predictor set to avoid target leakage.


In [ ]:
features = [
    "state", "time_of_day",
    "rain_or_wet", "fog", "night_evidence", "highway",
    "bus_involved", "lorry_truck", "car_involved",
    "two_wheeler", "head_on", "rear_end", "collision"
]

X = pilot[features].copy()
y = pilot["severity"].copy()

categorical = ["state", "time_of_day"]
numeric = [c for c in features if c not in categorical]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", "passthrough", numeric)
])


## 8. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))


## 9. Train four classification models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced"
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=5,
        class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=3,
        class_weight="balanced", random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, max_depth=3, random_state=42
    )
}

results = []
predictions = {}

for name, model in models.items():
    pipe = Pipeline([
        ("pre", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    predictions[name] = pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Macro Precision": precision_score(y_test, pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df.sort_values("Macro F1", ascending=False)


## 10. Model performance visualization

In [ ]:
plot_df = results_df.set_index("Model")[["Accuracy", "Macro F1"]]
plot_df.plot(kind="bar")
plt.title("Preliminary Model Performance")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 0.5)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 11. Decision Tree confusion matrix

In [ ]:
labels = ["Fatal", "Serious", "Minor/Reported Injury"]

cm = confusion_matrix(
    y_test,
    predictions["Decision Tree"],
    labels=labels
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)
disp.plot()
plt.title("Decision Tree Confusion Matrix – Pilot Test Set")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 12. Evaluation summary

In [ ]:
print(results_df.to_string(index=False))

best = results_df.loc[results_df["Macro F1"].idxmax()]
print("\nHighest pilot Macro F1:")
print(best)

print("\nDecision Tree classification report:")
print(classification_report(
    y_test,
    predictions["Decision Tree"],
    labels=labels,
    zero_division=0
))


## 13. Interpretation and limitations

The Decision Tree has the highest macro F1 in this saved pilot experiment (0.391) and accuracy of 0.414. This is a preliminary article-level pilot result.

The source corpus is based on public news reports. Multiple articles may describe the same accident, publication date may differ from accident date, and weather/road details may be missing. Therefore, the model should not be presented as a final operational accident-event predictor until event-level deduplication and validation are completed.
